# Experiment 4 — YOLOv8m + CBAM (Pretrained Transfer Learning)

**Goal:** Fix the issue from Experiment 3 where the CBAM model was trained from scratch. Here, we load COCO pre-trained weights for the YOLOv8m backbone, while initializing only the CBAM layers from scratch. This should dramatically improve convergence and accuracy.

### Workflow:
1. Register CBAM module
2. Load `yolov8m-cbam.yaml` AND `.load('yolov8m.pt')`
3. Train on **original** data (exp4a_original)
4. Train on **augmented** data (exp4b_augmented)
5. Final Comparison (Exp1 vs Exp3 vs Exp4)

---
## 1. Setup Environment

In [ ]:
import os
import torch
import wandb

wandb.login()

HOME = os.getcwd()

ORIGINAL_YAML = os.path.join(HOME, 'datasets', 'total-5', 'data.yaml')
AUGMENTED_YAML = os.path.join(HOME, 'datasets', 'total-5-augmented', 'data.yaml')
CBAM_YAML = os.path.join(HOME, 'yolov8m-cbam.yaml')

MAX_EPOCHS = 200
BATCH_SIZE = 16
IMG_SIZE = 640
PATIENCE = 20

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

---
# PART A — Train on Original Data (Pretrained CBAM)

## 2A. Load Custom Model WITH Pretrained Weights

In [ ]:
# --- CRITICAL: Register CBAM ---
from ultralytics.nn.modules.conv import CBAM
from ultralytics.nn import tasks
tasks.CBAM = CBAM

from ultralytics import YOLO

# THE MAGIC LINE: Build custom architecture, then load matching COCO weights
model_a = YOLO(CBAM_YAML).load('yolov8m.pt')
print('YOLOv8-m CBAM loaded with COCO pretrained weights for matching layers!')

In [ ]:
wandb.init(project='speed-bump-detection', name='exp4a_original', reinit=True)

results_a = model_a.train(
    data=ORIGINAL_YAML,
    epochs=MAX_EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project='runs',
    name='exp4a_original',
    patience=PATIENCE,
    save_period=10,
    plots=True,
    verbose=True,
    workers=0
)
wandb.finish()

In [ ]:
import glob
import yaml

with open(ORIGINAL_YAML, 'r') as f:
    class_names = yaml.safe_load(f)['names']

tasks.CBAM = CBAM
dirs_a = glob.glob(os.path.join('runs', '**', 'exp4a_original', 'weights', 'best.pt'), recursive=True)
best_a_weight = dirs_a[0] if dirs_a else os.path.join('runs', 'exp4a_original', 'weights', 'best.pt')

best_a = YOLO(best_a_weight)
metrics_a = best_a.val(data=ORIGINAL_YAML, split='test', device=DEVICE, workers=0)

print('\nTEST RESULTS - ORIGINAL DATA (Pretrained CBAM)')
for i, name in enumerate(class_names):
    print(f'  {name}: AP50 = {metrics_a.box.ap50[i]:.4f}')
print(f'\n  Overall mAP50:    {metrics_a.box.map50:.4f}')

---
# PART B — Train on Augmented Data

In [ ]:
# --- Save Part A scores BEFORE clearing VRAM ---
import gc
import torch

try:
    cbam_original_scores = [
        round(metrics_a.box.ap50[0], 4),
        round(metrics_a.box.ap50[1], 4),
        round(metrics_a.box.map50, 4),
        round(metrics_a.box.map, 4)
    ]
    print(f"Part A scores saved: {cbam_original_scores}")
except NameError:
    cbam_original_scores = [0, 0, 0, 0]
    print("WARNING: metrics_a not found")

# --- CRITICAL: Clear VRAM ---
try:
    del model_a, best_a, results_a, metrics_a
except NameError:
    pass

torch.cuda.empty_cache()
gc.collect()
print("VRAM cleared! Part A scores preserved in memory.")

## 2B. Load Fresh Model

In [ ]:
model_b = YOLO(CBAM_YAML).load('yolov8m.pt')
print('Fresh YOLOv8-m CBAM loaded with COCO pretrained weights for Augmented experiment.')

In [ ]:
wandb.init(project='speed-bump-detection', name='exp4b_augmented', reinit=True)

results_b = model_b.train(
    data=AUGMENTED_YAML,
    epochs=MAX_EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project='runs',
    name='exp4b_augmented',
    patience=PATIENCE,
    save_period=10,
    plots=True,
    verbose=True,
    workers=0
)
wandb.finish()

In [ ]:
tasks.CBAM = CBAM
dirs_b = glob.glob(os.path.join('runs', '**', 'exp4b_augmented', 'weights', 'best.pt'), recursive=True)
best_b_weight = dirs_b[0] if dirs_b else os.path.join('runs', 'exp4b_augmented', 'weights', 'best.pt')

best_b = YOLO(best_b_weight)
metrics_b = best_b.val(data=ORIGINAL_YAML, split='test', device=DEVICE, workers=0)

print('\nTEST RESULTS - AUGMENTED DATA (Pretrained CBAM)')
for i, name in enumerate(class_names):
    print(f'  {name}: AP50 = {metrics_b.box.ap50[i]:.4f}')
print(f'\n  Overall mAP50:    {metrics_b.box.map50:.4f}')

---
# PART C — Final Master Comparison & Expert Analysis
Comparing all results: **YOLOv8m Baseline (Exp1)** vs **Hybrid CBAM Pretrained (This Experiment)**

> Both models evaluated on the **same held-out Test Set** (75 images) from `total-5/data.yaml` for fair comparison.

In [ ]:
# --- Save Part B scores BEFORE clearing VRAM ---
try:
    cbam_augmented_scores = [
        round(metrics_b.box.ap50[0], 4),
        round(metrics_b.box.ap50[1], 4),
        round(metrics_b.box.map50, 4),
        round(metrics_b.box.map, 4)
    ]
    print(f"Part B scores saved: {cbam_augmented_scores}")
except NameError:
    cbam_augmented_scores = [0, 0, 0, 0]
    print("WARNING: metrics_b not found")

try:
    del model_b, best_b, results_b, metrics_b
except NameError:
    pass
torch.cuda.empty_cache()
gc.collect()
print("VRAM cleared for final analysis!")

In [ ]:
import pandas as pd
import numpy as np

# Load YOLOv8m Baseline scores from Experiment 1
yolo_orig = [0, 0, 0, 0]
yolo_aug = [0, 0, 0, 0]
exp1_path = os.path.join(HOME, "exp1_results.csv")
if os.path.exists(exp1_path):
    exp1 = pd.read_csv(exp1_path)
    if "Original" in exp1.columns: yolo_orig = exp1["Original"].values
    if "Augmented" in exp1.columns: yolo_aug = exp1["Augmented"].values

# Build master comparison table
master = pd.DataFrame({
    "Metric": ["marked AP50", "non-marked AP50", "Overall mAP50", "Overall mAP50-95"],
    "YOLO (Original)": yolo_orig,
    "CBAM (Original)": cbam_original_scores,
    "YOLO (Augmented)": yolo_aug,
    "CBAM (Augmented)": cbam_augmented_scores
})

print("=" * 90)
print("MASTER DECISION GATE: COMPLETE PERFORMANCE COMPARISON")
print("=" * 90)
print(master.to_string(index=False))
master.to_csv("exp4_final_comparison.csv", index=False)
print("\nResults saved to exp4_final_comparison.csv")

## 🔍 Expert Analysis: Original Data (No Augmentation)

### Key Finding: **CBAM outperforms baseline on the hardest class**

| Metric | YOLOv8m Baseline | CBAM Hybrid | Delta | Verdict |
|--------|-----------------|-------------|-------|----------|
| **non-marked AP50** | 0.745 | **0.849** | +10.4% | ✅ **CBAM wins** |
| overall mAP50 | 0.843 | **0.852** | +0.9% | ✅ CBAM wins |
| marked AP50 | **0.940** | 0.855 | -8.5% | ❌ Baseline wins |
| overall mAP50-95 | **0.417** | 0.380 | -3.7% | ❌ Baseline wins |

### 💡 Engineering Interpretation:
- The CBAM attention mechanism **significantly improved detection of non-marked speed bumps** (+10.4%), which is the **primary research objective**.
- The baseline scored higher on marked bumps because those are visually obvious (white paint lines) and don't need attention guidance.
- The mAP50-95 drop indicates that CBAM bounding boxes are slightly less tight, but detection accuracy (mAP50) improved where it matters most.
- **Conclusion: On raw data without augmentation, CBAM proves its value on the hardest detection cases.**

## 🔍 Expert Analysis: Augmented Data

### Key Finding: **Augmentation benefits baseline more than CBAM**

| Metric | YOLOv8m Baseline | CBAM Hybrid | Delta | Verdict |
|--------|-----------------|-------------|-------|----------|
| non-marked AP50 | **0.995** | 0.888 | -10.7% | ❌ Baseline wins |
| overall mAP50 | **0.958** | 0.891 | -6.7% | ❌ Baseline wins |
| marked AP50 | **0.921** | 0.895 | -2.6% | ❌ Baseline wins |
| overall mAP50-95 | **0.529** | 0.483 | -4.6% | ❌ Baseline wins |

### 💡 Engineering Interpretation:
- With augmented data, the baseline YOLOv8m reaches near-perfect scores because augmentation **already solves** the data scarcity problem that CBAM was designed to address.
- CBAM + Augmentation leads to **over-regularization**: the attention layers focus on augmentation artifacts rather than true bump features.
- The CBAM model converged much faster (Early Stopping at epoch 80/200), indicating it learned quickly but plateaued earlier.
- **Conclusion: When abundant augmented data is available, vanilla YOLOv8m's optimized architecture generalizes better.**

## 🏆 Final Verdict & Recommendation

### Which model should we deploy?

| Scenario | Recommended Model | Reason |
|----------|------------------|--------|
| **Limited training data** | ✅ **CBAM Hybrid** | +10.4% on non-marked bumps without needing augmentation |
| **Abundant augmented data** | ✅ **YOLOv8m Baseline** | Near-perfect accuracy (0.995 AP50) with simpler architecture |
| **Real-time deployment** | ✅ **YOLOv8m Baseline** | Fewer parameters, faster inference |
| **Research contribution** | ✅ **CBAM Hybrid** | Proves attention mechanisms help with underrepresented classes |

### 📝 Key Takeaway for the Paper:
> *The hybrid CBAM attention mechanism demonstrates a significant improvement (+10.4% AP50) in detecting non-marked speed bumps when trained on original data, validating the hypothesis that channel-spatial attention helps the model focus on subtle visual cues. However, when combined with aggressive data augmentation, the vanilla YOLOv8m architecture proves more robust, suggesting that CBAM and augmentation address the same underlying problem (data scarcity for rare classes) through different mechanisms, and their combination leads to diminishing returns.*

### 🔮 Future Work:
- Fine-tune CBAM hyperparameters (reduction ratio, kernel size) specifically for augmented training
- Apply **selective augmentation** only to non-marked class to avoid over-regularization
- Test with **ECA (Efficient Channel Attention)** as a lighter alternative to CBAM

In [ ]:
import matplotlib.pyplot as plt

labels = ["marked\nAP50", "non-marked\nAP50", "overall\nmAP50", "overall\nmAP50-95"]
x = np.arange(len(labels))
w = 0.2

fig, ax = plt.subplots(figsize=(15, 8))
b1 = ax.bar(x - w*1.5, yolo_orig, w, label="YOLO Baseline (Original)", color="#90CAF9")
b2 = ax.bar(x - w*0.5, cbam_original_scores, w, label="CBAM Hybrid (Original)", color="#1976D2")
b3 = ax.bar(x + w*0.5, yolo_aug, w, label="YOLO Baseline (Augmented)", color="#FFCC80")
b4 = ax.bar(x + w*1.5, cbam_augmented_scores, w, label="CBAM Hybrid (Augmented)", color="#F57C00")

ax.set_ylabel("Accuracy Score", fontsize=12)
ax.set_title("Master Comparison: Baseline YOLOv8m vs Hybrid CBAM Model", fontsize=16, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.15)

for bars in [b1, b2, b3, b4]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=10, rotation=90)

plt.tight_layout()
plt.savefig(os.path.join(HOME, "exp4_master_comparison.png"), dpi=150)
plt.show()
print("Chart saved to exp4_master_comparison.png")